# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim


# Device Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

Using Device: cpu


# Load Dataset

In [ ]:
df = pd.read_csv('/content/diabetes.csv')

print(df.head())
print("\nDataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset Shape: (768, 9)

Missing Values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome             

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [ ]:
# All columns except last one = features
X = df.iloc[:, :-1].values

# Last column (Outcome: 0 or 1) = target
y = df.iloc[:, -1].values

# Split into Train and Test Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert NumPy Arrays to PyTorch Tensors

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
# Create Dataset and DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# Build the Neural Network Model

class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()

        self.layer1 = nn.Linear(X_train.shape[1], 16)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

In [ ]:
model = DiabetesModel()

criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the Model

epochs = 50

for epoch in range(epochs):
    for inputs, labels in train_loader:

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 0.6909
Epoch [2/50], Loss: 0.6822
Epoch [3/50], Loss: 0.6774
Epoch [4/50], Loss: 0.6367
Epoch [5/50], Loss: 0.5445
Epoch [6/50], Loss: 0.7397
Epoch [7/50], Loss: 0.5318
Epoch [8/50], Loss: 0.3355
Epoch [9/50], Loss: 0.7896
Epoch [10/50], Loss: 0.4828
Epoch [11/50], Loss: 0.5249
Epoch [12/50], Loss: 0.3656
Epoch [13/50], Loss: 0.2975
Epoch [14/50], Loss: 0.5459
Epoch [15/50], Loss: 0.2314
Epoch [16/50], Loss: 0.4209
Epoch [17/50], Loss: 0.3017
Epoch [18/50], Loss: 0.4733
Epoch [19/50], Loss: 0.2720
Epoch [20/50], Loss: 0.5364
Epoch [21/50], Loss: 0.2532
Epoch [22/50], Loss: 0.2885
Epoch [23/50], Loss: 0.3876
Epoch [24/50], Loss: 0.1640
Epoch [25/50], Loss: 0.6495
Epoch [26/50], Loss: 0.2107
Epoch [27/50], Loss: 0.5704
Epoch [28/50], Loss: 0.3807
Epoch [29/50], Loss: 0.7497
Epoch [30/50], Loss: 0.3848
Epoch [31/50], Loss: 0.2061
Epoch [32/50], Loss: 0.2089
Epoch [33/50], Loss: 0.5529
Epoch [34/50], Loss: 0.7291
Epoch [35/50], Loss: 0.2367
Epoch [36/50], Loss: 0.2865
E

In [ ]:
# Test the Model (Calculate Accuracy)

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predicted = (outputs > 0.5).float()

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 77.27272727272727


# Email Spam DataSet

In [ ]:
import pandas as pd                     # For loading and handling dataset
import numpy as np                      # For numerical operations
import torch                            # Main PyTorch library
import torch.nn as nn                   # Neural network module
import torch.optim as optim             # Optimizers (Adam, SGD etc.)
from sklearn.model_selection import train_test_split  # For splitting dataset
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text to numbers
from sklearn.preprocessing import LabelEncoder         # Convert labels to numeric
from sklearn.metrics import accuracy_score             # Calculate accuracy


In [ ]:
df = pd.read_csv('/content/spam.csv', encoding='latin-1')  # Load dataset

# Keep only required columns (common spam dataset structure)
df = df[['v1', 'v2']]   # v1 = label (spam/ham), v2 = message
df.columns = ['label', 'message']  # Rename columns


In [ ]:
# ENCODE LABELS (spam=1, ham=0)
label_encoder = LabelEncoder()  # Create label encoder
df['label'] = label_encoder.fit_transform(df['label'])  # Convert text labels to numerical

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)  # Convert text to TF-IDF features (limit 5000 words)
X = vectorizer.fit_transform(df['message']).toarray()  # Transform messages into numeric matrix
y = df['label'].values  # Extract labels


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)  # 80% training, 20% testing


In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


/tmp/ipython-input-3262617754.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipython-input-3262617754.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
/tmp/ipython-input-3262617754.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
/tmp/ipython-input-3262617754.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or s

In [ ]:
class SpamClassifier(nn.Module):  # Define custom neural network class
    def __init__(self, input_size):  # Constructor
        super(SpamClassifier, self).__init__()  # Initialize parent class

        self.fc1 = nn.Linear(input_size, 128)  # First fully connected layer
        self.relu1 = nn.ReLU()                 # ReLU activation

        self.fc2 = nn.Linear(128, 64)          # Second hidden layer
        self.relu2 = nn.ReLU()                 # ReLU activation

        self.fc3 = nn.Linear(64, 1)            # Output layer (1 neuron for binary)
        self.sigmoid = nn.Sigmoid()            # Sigmoid activation for probability output

    def forward(self, x):   # Forward pass
        x = self.fc1(x)     # Pass through first layer
        x = self.relu1(x)   # Apply activation

        x = self.fc2(x)     # Second layer
        x = self.relu2(x)   # Activation

        x = self.fc3(x)     # Output layer
        x = self.sigmoid(x) # Convert to probability

        return x


In [ ]:
input_size = X_train.shape[1]   # Number of features
model = SpamClassifier(input_size)  # Create model instance


In [ ]:
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer


In [ ]:
epochs = 10  # Number of training iterations

for epoch in range(epochs):

    model.train()  # Set model to training mode

    optimizer.zero_grad()  # Clear previous gradients

    outputs = model(X_train).squeeze()  # Forward pass

    loss = criterion(outputs, y_train)  # Compute loss

    loss.backward()  # Backpropagation

    optimizer.step()  # Update weights

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.6393
Epoch [2/10], Loss: 0.6332
Epoch [3/10], Loss: 0.6268
Epoch [4/10], Loss: 0.6201
Epoch [5/10], Loss: 0.6130
Epoch [6/10], Loss: 0.6055
Epoch [7/10], Loss: 0.5974
Epoch [8/10], Loss: 0.5888
Epoch [9/10], Loss: 0.5797
Epoch [10/10], Loss: 0.5700


In [ ]:
model.eval()  # Set model to evaluation mode

with torch.no_grad():  # Disable gradient calculation

    test_outputs = model(X_test).squeeze()  # Get predictions
    predicted = (test_outputs >= 0.5).float()  # Convert probabilities to 0/1

    accuracy = accuracy_score(y_test, predicted)  # Calculate accuracy

print("Test Accuracy Before Saving:", accuracy)



Test Accuracy Before Saving: 0.8654708520179372


In [ ]:
torch.save(model.state_dict(), "spam_ann_model.pth")  # Save model weights

print('spam_ann_model.pth')

spam_ann_model.pth


In [ ]:
loaded_model = SpamClassifier(input_size)  # Create new model instance
loaded_model.load_state_dict(torch.load("spam_ann_model.pth"))  # Load weights
loaded_model.eval()  # Set to evaluation mode

SpamClassifier(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [ ]:
with torch.no_grad():

    loaded_outputs = loaded_model(X_test).squeeze()
    loaded_predicted = (loaded_outputs >= 0.5).float()

    loaded_accuracy = accuracy_score(y_test, loaded_predicted)

print("Test Accuracy After Loading:", loaded_accuracy)

Test Accuracy After Loading: 0.8654708520179372
